In [ ]:
from pathlib import Path
from netCDF4 import Dataset
import wrf
import xarray as xr
import numpy as np
import cartopy.crs as crs
import cartopy.feature as cfeature
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt

In [ ]:
d01data = Path.home() / "Projects/dyndowndata/Icestorm2021/wrf_d01/"
d02data = Path.home() / "Projects/dyndowndata/Icestorm2021/wrf_d02/"
wrfdata = Path.home() / "Projects/dyndowndata/Icestorm2021/" 

### Plot wind speed and surface-level pressure for a test file

Let's look at a 12 km resolution file first.

In [ ]:
testfile = d01data / "wrfout_d01_2021-12-25_18:00:00"
ncfile = Dataset(testfile)

In [ ]:
uvmet10 = wrf.getvar(ncfile, "uvmet10", timeidx=wrf.ALL_TIMES, method='cat')
u10 = uvmet10.sel(u_v='u', drop=True)
u10

In [ ]:
slp = wrf.getvar(ncfile, "slp", timeidx=0)
wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=0)
rainnc = wrf.getvar(ncfile, "RAINNC", timeidx=0)
lats, lons = wrf.latlon_coords(slp)
cart_proj = wrf.get_cartopy(slp)

In [ ]:
wrflist = [
        Dataset(item) for item in sorted(list((wrfdata / f"211225").glob("wrfout_d01*")))
    ]
slp = wrf.getvar(wrflist, "slp", timeidx=wrf.ALL_TIMES, method='cat')

In [ ]:
slp.sel(Time=slice('2021-12-26', '2021-12-26'))

In [ ]:
for idx in range(6):
    wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=idx)
    wind.sel(wspd_wdir='wspd').plot(levels=10)
    plt.show()

In [ ]:
for idx in range(6):
    slp = wrf.getvar(ncfile, "slp", timeidx=idx)
    slp.plot(levels=15)
    plt.show()

Now for a 4 km resolution file.

At a location

In [ ]:
day = 25
fbx_lon, fbx_lat = -147.7200, 64.8401
testfile = d02data / f"wrfout_d02_2021-12-{day}_00:00:00"
with Dataset(testfile) as src:
    xx, yy = wrf.ll_to_xy(src, fbx_lat, fbx_lon, as_int=True).data
print(xx, yy, fbx_lat, fbx_lon)

for testfile in sorted(list((wrfdata / f"2112{day}").glob("wrfout_d02*"))):
    print(testfile.stem)

    with Dataset(testfile) as src:
        rainc = wrf.getvar(src, 'RAINC', timeidx=wrf.ALL_TIMES)
        rainnc = wrf.getvar(src, 'RAINNC', timeidx=wrf.ALL_TIMES)
    print(rainnc.isel(west_east=xx, south_north=yy).data )

In [ ]:
day = 25
fbx_lon, fbx_lat = -147.7200, 64.8401
testfile = d02data / f"wrfout_d02_2021-12-{day}_00:00:00"
with Dataset(testfile) as src:
    xx, yy = wrf.ll_to_xy(src, fbx_lat, fbx_lon, as_int=True).data
print(xx, yy, fbx_lat, fbx_lon)

rainlist = []
for day in [21, 23, 25, 27, 29]:
    wrflist = [
        Dataset(item) for item in sorted(list((wrfdata / f"2112{day}").glob("wrfout_d02*")))
    ]
    rainnc = wrf.getvar(wrflist, 'RAINNC', timeidx=wrf.ALL_TIMES, method="cat")
    rainc = wrf.getvar(wrflist, 'RAINC', timeidx=wrf.ALL_TIMES, method="cat")
    rain_delta_FBKS = ( 
        rainnc.diff("Time").isel(west_east=xx, south_north=yy, Time=slice(6, None)) +
        rainc.diff("Time").isel(west_east=xx, south_north=yy, Time=slice(6, None)) )
    # print(rain_delta_FBKS)
    rainlist.append(rain_delta_FBKS)

rain = xr.concat(rainlist, dim='Time')


rain.plot()
plt.title("Fairbanks, RAINC + RAINNC")
plt.ylabel("mm")

In [ ]:
ax = plt.axes()
rain.plot(label='total precip', ax=ax, lw=0.8, ls=':')
(rain - snow).plot(label='rain only', ax=ax)
plt.title("Fairbanks, rain storm 2021")
plt.legend()
plt.ylabel("mm")

In [ ]:
levels = np.arange(0, 5, 10)

In [ ]:

day = 25
fbx_lon, fbx_lat = -147.7200, 64.8401
testfile = d02data / f"wrfout_d02_2021-12-{day}_00:00:00"
with Dataset(testfile) as src:
    xx, yy = wrf.ll_to_xy(src, fbx_lat, fbx_lon, as_int=True).data
print(xx, yy, fbx_lat, fbx_lon)

projection = crs.AlbersEqualArea(
    central_longitude=-154.0, central_latitude=50.0, 
    standard_parallels=(55.0, 65.0))

for day in [25, 27, 29]:
    wrflist = [
        Dataset(item) for item in sorted(list((wrfdata / f"2112{day}").glob("wrfout_d02*")))
    ]
    rainnc = wrf.getvar(wrflist, 'RAINNC', timeidx=wrf.ALL_TIMES, method="cat")
    rainc = wrf.getvar(wrflist, 'RAINC', timeidx=wrf.ALL_TIMES, method="cat")
    rain_delta = ( 
        rainnc.diff("Time").isel(Time=slice(6, None)) +
        rainc.diff("Time").isel(Time=slice(6, None)) )
    rain_delta = rain_delta.where(rain_delta >= 0, other=0)
    rain_delta = rain_delta.where(rain_delta <= 10, other=9.9999)
    # print(rain_delta.max())
    for idx in range(48):
        fig = plt.figure(figsize=(10,8))
        ax = plt.axes(projection=projection)
        ax.set_extent([-166, -136, 52.5, 72])
        # ax.set_extent([-160, -140, 60, 68])
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS)
        rain_delta.isel(Time=idx).plot(vmin=0, vmax=10, levels=11, cmap="cubehelix_r", ax=ax, transform=crs.PlateCarree(),
            x="XLONG", y="XLAT", cbar_kwargs={'label': "mm"})
        plt.title(f"Ice storm precip hourly rain, {rain_delta.isel(Time=idx).Time.dt.strftime('%Y-%m-%d %H:%M').data} UTC")
        plt.show()
        fig.savefig(f"Icestorm2021_{rain_delta.isel(Time=idx).Time.dt.strftime('%Y%m%d_%H%M_UTC').data}.png", bbox_inches='tight')



In [ ]:

day = 25
fbx_lon, fbx_lat = -147.7200, 64.8401
testfile = d02data / f"wrfout_d02_2021-12-{day}_00:00:00"
with Dataset(testfile) as src:
    xx, yy = wrf.ll_to_xy(src, fbx_lat, fbx_lon, as_int=True).data
print(xx, yy, fbx_lat, fbx_lon)

projection = crs.AlbersEqualArea(
    central_longitude=-154.0, central_latitude=50.0, 
    standard_parallels=(55.0, 65.0))

for day in [25, 27, 29]:
    wrflist = [
        Dataset(item) for item in sorted(list((wrfdata / f"2112{day}").glob("wrfout_d02*")))
    ]
    rainnc = wrf.getvar(wrflist, 'RAINNC', timeidx=wrf.ALL_TIMES, method="cat")
    rainc = wrf.getvar(wrflist, 'RAINC', timeidx=wrf.ALL_TIMES, method="cat")
    acsnow = wrf.getvar(wrflist, 'ACSNOW', timeidx=wrf.ALL_TIMES, method="cat")
    rain_only = (rainc + rainnc) - acsnow
    rain_delta = rain_only.diff("Time").isel(Time=slice(6, None)) 
    rain_delta = rain_delta.where(rain_delta >= 0, other=0)
    rain_delta = rain_delta.where(rain_delta <= 10, other=9.9999)
    # print(rain_delta.max())
    for idx in range(48):
        fig = plt.figure(figsize=(10,8))
        ax = plt.axes(projection=projection)
        ax.set_extent([-166, -136, 52.5, 72])
        # ax.set_extent([-160, -140, 60, 68])
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS)
        rain_delta.isel(Time=idx).plot(vmin=0, vmax=10, levels=11, cmap="cubehelix_r", ax=ax, transform=crs.PlateCarree(),
            x="XLONG", y="XLAT", cbar_kwargs={'label': "mm"})
        plt.title(f"Ice storm rain only (hourly), {rain_delta.isel(Time=idx).Time.dt.strftime('%Y-%m-%d %H:%M').data} UTC")
        plt.show()
        fig.savefig(f"Icestorm2021_rainonly_{rain_delta.isel(Time=idx).Time.dt.strftime('%Y%m%d_%H%M_UTC').data}.png", bbox_inches='tight')

In [ ]:
rain_only.min()

In [ ]:
rain_delta = rain_delta.where(rain_delta >= 0, other=0)

for idx in range(46, 48):
    ax = plt.axes(projection=projection)
    ax.set_extent([-166, -135, 52.5, 72])
    ax.coastlines(color='#111', lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.RIVERS)
    rain_delta.isel(Time=idx).plot(vmin=0, vmax=10, levels=15, cmap="cubehelix_r", ax=ax, transform=crs.PlateCarree(),
    x="XLONG", y="XLAT")
    plt.show()

In [ ]:
day = 25
fbx_lon, fbx_lat = -147.7200, 64.8401
testfile = d02data / f"wrfout_d02_2021-12-{day}_00:00:00"
with Dataset(testfile) as src:
    xx, yy = wrf.ll_to_xy(src, fbx_lat, fbx_lon, as_int=True).data
print(xx, yy, fbx_lat, fbx_lon)

snowlist = []
for day in [21, 23, 25, 27, 29]:
    wrflist = [
        Dataset(item) for item in sorted(list((wrfdata / f"2112{day}").glob("wrfout_d02*")))
    ]
    acsnow = wrf.getvar(wrflist, 'ACSNOW', timeidx=wrf.ALL_TIMES)
    snow_delta_FBKS = acsnow.diff("Time").isel(west_east=xx, south_north=yy, Time=slice(6, None)) 
    # print(snow_delta_FBKS)
    snowlist.append(snow_delta_FBKS)

snow = xr.concat(snowlist, dim='Time')


snow.plot()
plt.title("Fairbanks, ACSNOW hourly")
plt.ylabel("kg m-2")


In [ ]:
acsnow.isel(Time=5).plot()

### Make animated GIF

In [ ]:
from PIL import Image
import glob

In [ ]:
# Create the frames
frames = []
imgs = sorted(list(glob.glob("Icestorm2021_rainonly*.png")))
frames = [Image.open(img) for img in imgs]
 
# Save into a GIF file that loops forever
frames[0].save('Icestorm2021_rainonly_d02.gif', format='GIF',
               append_images=frames[1:],
               save_all=True,
               duration=300, loop=0)